# Optuna Hyperparameter Search — XGBoost Baseline

Automated search over XGBoost hyperparameters using Optuna (TPE sampler + MedianPruner).
Forked from `xgb_baseline_0410.ipynb` and structured like `optuna_hpsearch_0409.ipynb`.

Sequence flattening is deterministic and trial-independent, so we flatten once outside
the objective and reuse the same `DMatrix` across all trials.

In [ ]:
import os
import gc
import json
import datetime

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score

import optuna
from optuna.integration import XGBoostPruningCallback

print(f"XGBoost version: {xgb.__version__}")
print(f"Optuna version:  {optuna.__version__}")

## Fixed Configuration

In [ ]:
# ---------------------------------------------------------------------------
# Feature schema — FIXED (not tuned), must match train_df / test_df columns
# ---------------------------------------------------------------------------
CATEGORICAL_FEATURES = [
    "sess_pid_seq",
    "sess_csid_seq",
    "sess_ccid_seq",
    "sess_bid_seq",
]

CONTINUOUS_FEATURES = [
    "sess_price_log_norm_seq",
    "sess_dtime_secs_log_norm_seq",
    "sess_prod_recency_days_log_norm_seq",
]

STATIC_CATEGORICAL_FEATURES = [
    "user_segment",
    "device_type",
]

STATIC_CONTINUOUS_FEATURES = [
    "user_lifetime_days",
    "user_avg_order_value",
]

TARGET_COLUMN = "target"

# ---------------------------------------------------------------------------
# Flattening / search configuration
# ---------------------------------------------------------------------------
MAX_SEQ_LEN = 20
PAD_VALUE = 0

N_TRIALS = 200
NUM_BOOST_ROUND = 500
EARLY_STOPPING_ROUNDS = 50

## Load Data

In [ ]:
# train_df = pd.read_parquet("train.parquet")
# test_df  = pd.read_parquet("test.parquet")

print(f"Train: {len(train_df)} sessions  (positives: {train_df[TARGET_COLUMN].mean():.4%})")
print(f"Test:  {len(test_df)} sessions  (positives: {test_df[TARGET_COLUMN].mean():.4%})")

## Flatten Once (shared across all trials)

In [ ]:
def _flatten_seq_column(series: pd.Series, dtype) -> np.ndarray:
    n = len(series)
    out = np.full((n, MAX_SEQ_LEN), PAD_VALUE, dtype=dtype)
    for i, seq in enumerate(series.values):
        arr = np.asarray(seq, dtype=dtype)[-MAX_SEQ_LEN:]
        k = len(arr)
        if k:
            out[i, MAX_SEQ_LEN - k:] = arr
    return out


def flatten_sessions(df: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for feat in CATEGORICAL_FEATURES:
        mat = _flatten_seq_column(df[feat], dtype=np.int64)
        cols = [f"{feat}_{i}" for i in range(MAX_SEQ_LEN)]
        frames.append(pd.DataFrame(mat, columns=cols, index=df.index))
    for feat in CONTINUOUS_FEATURES:
        mat = _flatten_seq_column(df[feat], dtype=np.float32)
        cols = [f"{feat}_{i}" for i in range(MAX_SEQ_LEN)]
        frames.append(pd.DataFrame(mat, columns=cols, index=df.index))
    if STATIC_CATEGORICAL_FEATURES:
        frames.append(df[STATIC_CATEGORICAL_FEATURES].astype(np.int64))
    if STATIC_CONTINUOUS_FEATURES:
        frames.append(df[STATIC_CONTINUOUS_FEATURES].astype(np.float32))
    return pd.concat(frames, axis=1)


def _seq_cat_columns(feat: str):
    return [f"{feat}_{i}" for i in range(MAX_SEQ_LEN)]


def all_categorical_columns():
    cols = []
    for feat in CATEGORICAL_FEATURES:
        cols.extend(_seq_cat_columns(feat))
    cols.extend(STATIC_CATEGORICAL_FEATURES)
    return cols


X_train = flatten_sessions(train_df)
y_train = train_df[TARGET_COLUMN].astype(np.float32).values
X_test  = flatten_sessions(test_df)
y_test  = test_df[TARGET_COLUMN].astype(np.float32).values

# ---------------------------------------------------------------------------
# Unseen-category sanity check: remap test values not seen in train → UNSEEN_FILL (= 0).
# See xgb_baseline_0410.ipynb for the detailed per-feature reporting of unseen values.
# ---------------------------------------------------------------------------
UNSEEN_FILL = PAD_VALUE
n_remapped_total = 0

for feat in CATEGORICAL_FEATURES:
    cols = _seq_cat_columns(feat)
    seen = np.unique(X_train[cols].to_numpy())
    arr = X_test[cols].to_numpy()
    mask = ~np.isin(arr, seen)
    if mask.any():
        arr = arr.copy()
        arr[mask] = UNSEEN_FILL
        X_test[cols] = arr
        n_remapped_total += int(mask.sum())

for feat in STATIC_CATEGORICAL_FEATURES:
    seen = np.unique(X_train[feat].to_numpy())
    vals = X_test[feat].to_numpy()
    mask = ~np.isin(vals, seen)
    if mask.any():
        vals = vals.copy()
        vals[mask] = UNSEEN_FILL
        X_test[feat] = vals
        n_remapped_total += int(mask.sum())

# Cast categoricals to a shared dtype (train ∪ {UNSEEN_FILL}).
for c in all_categorical_columns():
    cats = sorted(set(X_train[c].unique().tolist()) | {UNSEEN_FILL})
    dtype = pd.CategoricalDtype(categories=cats)
    X_train[c] = X_train[c].astype(dtype)
    X_test[c] = X_test[c].astype(dtype)

for c in all_categorical_columns():
    assert X_test[c].isna().sum() == 0, f"Unknown category leaked into {c} after remap"

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Remapped {n_remapped_total:,} unseen categorical values in test → {UNSEEN_FILL}")

dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
dtest  = xgb.DMatrix(X_test,  label=y_test,  enable_categorical=True)

## Optuna Objective

In [ ]:
def objective(trial):
    params = {
        "objective":        "binary:logistic",
        "eval_metric":      "auc",
        "tree_method":      "hist",
        "max_depth":        trial.suggest_int("max_depth", 3, 12),
        "learning_rate":    trial.suggest_float("learning_rate", 1e-3, 3e-1, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-2, 10.0, log=True),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "gamma":             trial.suggest_float("gamma", 1e-8, 5.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 5.0, 50.0),
    }
    num_boost_round = trial.suggest_int("num_boost_round", 100, NUM_BOOST_ROUND)

    pruning_cb = XGBoostPruningCallback(trial, "test-auc")

    booster = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=num_boost_round,
        evals=[(dtrain, "train"), (dtest, "test")],
        verbose_eval=False,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        callbacks=[pruning_cb],
    )

    best_auc = float(booster.best_score)
    trial.set_user_attr("best_iteration", int(booster.best_iteration + 1))

    del booster
    gc.collect()
    return best_auc

## Run Study

In [ ]:
time_stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")
study_name = f"xgb_hpsearch_{time_stamp}"

study = optuna.create_study(
    study_name=study_name,
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=10,
        n_warmup_steps=20,
    ),
    storage=f"sqlite:///{study_name}.db",
    load_if_exists=True,
)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nCompleted {len(study.trials)} trials")
print(f"Best AUC: {study.best_value:.6f}")
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print(f"  best_iteration (user_attr): {study.best_trial.user_attrs.get('best_iteration')}")

## Results Analysis

In [ ]:
trials_df = study.trials_dataframe()
trials_df = trials_df.sort_values("value", ascending=False)

display_cols = ["number", "value", "state"] + [c for c in trials_df.columns if c.startswith("params_")]
print("Top 10 trials:")
print(trials_df[display_cols].head(10).to_string())

results_path = f"optuna_xgb_results_{time_stamp}.csv"
trials_df.to_csv(results_path, index=False)
print(f"\nFull results saved to {results_path}")

## Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# --- 1. Optimization history ---
ax = axes[0, 0]
completed = trials_df[trials_df["state"] == "COMPLETE"].copy().sort_values("number")
ax.scatter(completed["number"], completed["value"], s=10, alpha=0.6)
running_best = completed["value"].cummax()
ax.plot(completed["number"], running_best, color="red", linewidth=2, label="Best so far")
ax.set_xlabel("Trial")
ax.set_ylabel("AUC")
ax.set_title("Optimization History")
ax.legend()
ax.grid(True, alpha=0.3)

# --- 2. Parameter importances ---
ax = axes[0, 1]
try:
    importances = optuna.importance.get_param_importances(study)
    names = list(importances.keys())[:12]
    values = [importances[n] for n in names]
    ax.barh(names[::-1], values[::-1])
    ax.set_xlabel("Importance")
    ax.set_title("Hyperparameter Importance")
except Exception as e:
    ax.text(0.5, 0.5, f"Could not compute\nimportances:\n{e}",
            ha="center", va="center", transform=ax.transAxes)

# --- 3. Learning rate vs AUC ---
ax = axes[1, 0]
if "params_learning_rate" in completed.columns:
    ax.scatter(completed["params_learning_rate"], completed["value"], s=10, alpha=0.5)
    ax.set_xscale("log")
    ax.set_xlabel("Learning Rate")
    ax.set_ylabel("AUC")
    ax.set_title("Learning Rate vs AUC")
    ax.grid(True, alpha=0.3)

# --- 4. scale_pos_weight vs AUC ---
ax = axes[1, 1]
if "params_scale_pos_weight" in completed.columns:
    ax.scatter(completed["params_scale_pos_weight"], completed["value"], s=10, alpha=0.5)
    ax.set_xlabel("scale_pos_weight")
    ax.set_ylabel("AUC")
    ax.set_title("scale_pos_weight vs AUC")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Export Best Hyperparameters

In [ ]:
best = study.best_trial

best_hp = {
    "timestamp": time_stamp,
    "study_name": study_name,
    "best_trial": best.number,
    "best_auc": best.value,
    "n_trials_completed": len([t for t in study.trials if t.state.name == "COMPLETE"]),
    "n_trials_pruned":    len([t for t in study.trials if t.state.name == "PRUNED"]),
    "best_iteration": best.user_attrs.get("best_iteration"),
    "hyperparameters": {
        "MAX_DEPTH":         best.params["max_depth"],
        "LEARNING_RATE":     best.params["learning_rate"],
        "SUBSAMPLE":         best.params["subsample"],
        "COLSAMPLE_BYTREE":  best.params["colsample_bytree"],
        "MIN_CHILD_WEIGHT":  best.params["min_child_weight"],
        "REG_ALPHA":         best.params["reg_alpha"],
        "REG_LAMBDA":        best.params["reg_lambda"],
        "GAMMA":             best.params["gamma"],
        "POS_WEIGHT":        best.params["scale_pos_weight"],
        "NUM_BOOST_ROUND":   best.params["num_boost_round"],
    },
}

hp_path = f"optuna_best_xgb_hp_{time_stamp}.json"
with open(hp_path, "w") as f:
    json.dump(best_hp, f, indent=2)

print(f"Best hyperparameters saved to {hp_path}")
print("\nCopy these into xgb_baseline_0410.ipynb config cell:")
print("=" * 60)
for k, v in best_hp["hyperparameters"].items():
    print(f"{k} = {v}")
print("=" * 60)